# ハイパーパラメータ調整

In [ ]:
import os
import warnings
from datetime import datetime

import tensorboard
from google.cloud import aiplatform
from google.cloud.aiplatform import hyperparameter_tuning as hpt


%load_ext tensorboard
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
# 必要に応じて以下を変更
PROJECT = !gcloud config get-value project  # noqa: E999
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION
%env TFVERSION=2.8

In [ ]:
%%bash
gcloud config set project $PROJECT
gcloud config set ai/region $REGION

## Agent Platform Training Service とコードを互換性のあるものにする

### Google Cloud Storage（GCS）へのデータのアップロード

In [ ]:
!gcloud storage ls gs://$BUCKET/taxifare/data

## コードを Python パッケージに移動する

In [ ]:
!ls -la taxifare/trainer

トレーニングジョブでハイパーパラメータ調整を使用するには、以下の手順を実行する必要があります。

1. ハイパーパラメータ調整の指標タグと目標（goal）を指定する
2. チューニングするハイパーパラメータを `task.py` に設定する
3. `model.py` でハイパーパラメータをモデルに渡す
4. 同じく `model.py` で、`hypertune` ライブラリを使用して指標を Agent Platform に報告する

### model.py の修正

#### 演習

以下の `taxifare/trainer/model.py` 内の TODO を完了してください。
 - `HPTCallback` 内で、`hpt.report_hyperparameter_tuning_metric` を使用して指標を報告する

In [ ]:
%%writefile ./taxifare/trainer/model.py
"""Data prep, train and evaluate DNN model."""

import logging
import os
import hypertune

import numpy as np
import tensorflow as tf
import keras
from keras import callbacks
from keras.layers import (
    Concatenate,
    Dense,
    Discretization,
    Embedding,
    Flatten,
    HashedCrossing,
    Input,
    Lambda,
)
from keras.metrics import RootMeanSquaredError

def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # Label: fare_amount
    label = tf.strings.to_number(ds[0])
    # Features: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    feature = tf.strings.to_number(ds[2:6])  # use some features only
    # Passing feature in tuple so that we can handle them separately.
    return (feature[0], feature[1], feature[2], feature[3]), label


def create_dataset(pattern, batch_size, num_repeat, mode="eval"):
    ds = tf.data.Dataset.list_files(pattern)
    ds = ds.flat_map(tf.data.TextLineDataset)
    ds = ds.map(parse_csv)
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.repeat(num_repeat).batch(batch_size, drop_remainder=True)
    return ds


def parse_lat_lon(row):
    columns = tf.strings.split(row, ",")
    # latitude idx: 3 and 5, longitude idx: 2 and 4
    lat_strings = tf.gather(columns, [3, 5])
    lon_strings = tf.gather(columns, [2, 4])
    lat_features = tf.strings.to_number(lat_strings)
    lon_features = tf.strings.to_number(lon_strings)
    return lat_features, lon_features


def adapt_normalize(train_data_path):
    ds = tf.data.Dataset.list_files(train_data_path)
    ds = ds.flat_map(tf.data.TextLineDataset)
    lat_lon_ds = ds.map(parse_lat_lon).batch(10000)

    lat_scaler = keras.layers.Normalization(axis=None)
    lon_scaler = keras.layers.Normalization(axis=None)

    lat_values, lon_values = next(iter(lat_lon_ds))

    lat_scaler.adapt(lat_values)
    lon_scaler.adapt(lon_values)

    print("Computed statistics for latitude:")
    print(f"mean: {lat_scaler.mean}, variance: {lat_scaler.variance}")
    print("+++++")
    print("Computed statistics for longitude:")
    print(f"mean: {lon_scaler.mean}, variance: {lon_scaler.variance}")

    return lat_scaler, lon_scaler


def euclidean(params):
    lon1, lat1, lon2, lat2 = params
    londiff = lon2 - lon1
    latdiff = lat2 - lat1
    return tf.sqrt(londiff * londiff + latdiff * latdiff)


def transform(inputs, nbuckets, normalizers):
    lat_scaler, lon_scaler = normalizers

    # 経度の正規化
    scaled_plon = lon_scaler(inputs["pickup_longitude"])
    scaled_dlon = lon_scaler(inputs["dropoff_longitude"])

    # 緯度の正規化
    scaled_plat = lat_scaler(inputs["pickup_latitude"])
    scaled_dlat = lat_scaler(inputs["dropoff_latitude"])

    # カスタムユークリッド関数のための Lambda レイヤー
    euclidean_distance = Lambda(euclidean, name="euclidean")(
        [scaled_plon, scaled_plat, scaled_dlon, scaled_dlat]
    )

    # 離散化（バケット化）
    latbuckets = np.linspace(start=-5, stop=5, num=nbuckets).tolist()
    lonbuckets = np.linspace(start=-5, stop=5, num=nbuckets).tolist()

    plon = Discretization(lonbuckets, name="plon_bkt")(scaled_plon)
    plat = Discretization(latbuckets, name="plat_bkt")(scaled_plat)
    dlon = Discretization(lonbuckets, name="dlon_bkt")(scaled_dlon)
    dlat = Discretization(latbuckets, name="dlat_bkt")(scaled_dlat)

    # HashedCrossing レイヤーを使用した特徴量クロス
    p_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 2, name="p_fc")((plon, plat))
    d_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 2, name="d_fc")((dlon, dlat))
    pd_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 4, name="pd_fc")((p_fc, d_fc))

    # Embedding レイヤーによる埋め込み
    pd_embed = Flatten()(
        Embedding(input_dim=(nbuckets + 1) ** 4, output_dim=10, name="pd_embed")(
            pd_fc
        )
    )

    transformed = Concatenate()([
        scaled_plon,
        scaled_dlon,
        scaled_plat,
        scaled_dlat,
        euclidean_distance, 
        pd_embed
    ])

    return transformed


def build_dnn_model(nbuckets, nnsize, lr, normalizers):
    INPUT_COLS = [
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
    ]

    inputs = {
        colname: Input(name=colname, shape=(1,), dtype="float32")
        for colname in INPUT_COLS
    }

    # 変換
    x = transform(inputs, nbuckets, normalizers)

    for layer, nodes in enumerate(nnsize):
        x = Dense(nodes, activation="relu", name=f"h{layer}")(x)
    output = Dense(1, name="fare")(x)

    model = keras.Model(inputs=list(inputs.values()), outputs=output)
    lr_optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=lr_optimizer, loss="mse", metrics=[RootMeanSquaredError()])

    return model

# TODO 1
hpt = # TODO: ここにコードを記述してください


# 報告用コールバック
# TODO 1
class HPTCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        global hpt
        # TODO: ここにコードを記述してください


def train_and_evaluate(hparams):
    batch_size = hparams["batch_size"]
    nbuckets = hparams["nbuckets"]
    lr = hparams["lr"]
    nnsize = [int(s) for s in hparams["nnsize"].split()]
    eval_data_path = hparams["eval_data_path"]
    num_evals = hparams["num_evals"]
    num_examples_to_train_on = hparams["num_examples_to_train_on"]
    output_dir = hparams["output_dir"]
    train_data_path = hparams["train_data_path"]

    model_export_path = os.path.join(output_dir, "model.keras")
    serving_model_export_path = os.path.join(output_dir, "savedmodel")
    checkpoint_path = os.path.join(output_dir, "checkpoint.keras")
    tensorboard_path = os.path.join(output_dir, "tensorboard")

    if tf.io.gfile.exists(output_dir):
        tf.io.gfile.rmtree(output_dir)

    normalizers = adapt_normalize(eval_data_path)

    model = build_dnn_model(nbuckets, nnsize, lr, normalizers)
    logging.info(model.summary())
    
    trainds = create_dataset(
        pattern=train_data_path, batch_size=batch_size, num_repeat=None, mode="train"
    )

    evalds = create_dataset(
        pattern=eval_data_path, batch_size=batch_size, num_repeat=1, mode="eval"
    )

    steps_per_epoch = num_examples_to_train_on // (batch_size * num_evals)

    checkpoint_cb = callbacks.ModelCheckpoint(
        checkpoint_path, verbose=1
    )
    tensorboard_cb = callbacks.TensorBoard(tensorboard_path, histogram_freq=1)

    history = model.fit(
        trainds,
        validation_data=evalds,
        epochs=num_evals,
        steps_per_epoch=max(1, steps_per_epoch),
        verbose=2,  # 0=silent, 1=progress bar, 2=one line per epoch
        callbacks=[checkpoint_cb, tensorboard_cb, HPTCallback()],
    )

    # Keras モデルファイルを保存。
    model.save(model_export_path)
    # サービング用に SavedModel 形式でモデルをエクスポート。
    model.export(serving_model_export_path)
    return history

### task.py の修正

In [ ]:
%%writefile taxifare/trainer/task.py
"""trainer.model のモデルトレーニングコード用の引数定義。"""
import argparse
import os

from trainer import model

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--batch_size",
        help="Batch size for training steps",
        type=int,
        default=32,
    )
    parser.add_argument(
        "--eval_data_path",
        help="GCS location pattern of eval files",
        required=True,
    )
    parser.add_argument(
        "--nnsize",
        help="Hidden layer sizes (provide space-separated sizes)",
        default="32 8",
    )
    parser.add_argument(
        "--nbuckets",
        help="Number of buckets to divide lat and lon with",
        type=int,
        default=10,
    )
    parser.add_argument(
        "--lr", help="learning rate for optimizer", type=float, default=0.001
    )
    parser.add_argument(
        "--num_evals",
        help="Number of times to evaluate model on eval data training.",
        type=int,
        default=5,
    )
    parser.add_argument(
        "--num_examples_to_train_on",
        help="Number of examples to train on.",
        type=int,
        default=100,
    )
    parser.add_argument(
        "--output_dir",
        help="GCS location to write checkpoints and export models",
        default=os.getenv("AIP_MODEL_DIR"),
    )
    parser.add_argument(
        "--train_data_path",
        help="GCS location pattern of train files containing eval URLs",
        required=True,
    )

    args, _ = parser.parse_known_args()

    hparams = args.__dict__
    print("output_dir", hparams["output_dir"])
    model.train_and_evaluate(hparams)


In [ ]:
%%writefile taxifare/setup.py
"""ソース配布物を作成するための setuptools の使用。"""

from setuptools import find_packages, setup

setup(
    name="taxifare_trainer",
    version="0.1",
    packages=find_packages(),
    include_package_data=True,
    description="Taxifare model training application.",
)

In [ ]:
%%bash
cd taxifare
python ./setup.py sdist --formats=gztar
cd ..

In [ ]:
%%bash
gcloud storage cp taxifare/dist/taxifare_trainer-0.1.tar.gz gs://${BUCKET}/taxifare/

## Agent Platform 上でのハイパーパラメータ調整ジョブの実行

In [ ]:
NUM_EXAMPLES_TO_TRAIN_ON = 50000
NUM_EVALS = 20
NNSIZE = "32 8"

base_path = f"gs://{BUCKET}/taxifare"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

args = [
    "--eval_data_path",
    f"{base_path}/data/taxi-valid*",
    "--train_data_path",
    f"{base_path}/data/taxi-train*",
    "--num_examples_to_train_on",
    f"{NUM_EXAMPLES_TO_TRAIN_ON}",
    "--num_evals",
    f"{NUM_EVALS}",
    "--nnsize",
    f"{NNSIZE}",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "python_package_spec": {
            "executor_image_uri": "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest",
            "package_uris": [f"{base_path}/taxifare_trainer-0.1.tar.gz"],
            "python_module": "trainer.task",
            "args": args,
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name="custom_job",
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{base_path}/staging",
    base_output_dir=f"{base_path}/trained_model_{timestamp}",
)

### HyperparameterTuningJob の作成

それでは、ハイパーパラメータ調整ジョブの仕様を定義しましょう。
ハイパーパラメータ調整ジョブを構成する際は、以下の詳細を指定する必要があります。

#### `metric_spec`
最適化のために使用する指標。辞書 `{<指標のタグ名>: <目標>}` で指定できます。

指標名は、トレーニングアプリケーションで報告するタグ名に対応している必要があります。

目標（goal）には、`'maximize'`（例: 精度）または `'minimize'`（例: 損失値）のいずれかを設定できます。

#### `parameter_spec`
ParameterSpec オブジェクトでは、ハイパーパラメータのデータ型をパラメータ値仕様のインスタンスとして指定します。以下の表に、サポートされているパラメータ値仕様を示します。

|Type|Data type|Value ranges|Value data|
|--|--|--|--|
|DoubleValueSpec|DOUBLE|minValue & maxValue|Floating-point values（浮動小数点値）|
|IntegerValueSpec|INTEGER|minValue & maxValue|Integer values（整数値）|
|CategoricalValueSpec|CATEGORICAL|categoricalValues|List of category strings（カテゴリ文字列のリスト）|
|DiscreteValueSpec|DISCRETE|discreteValues|List of values in ascending order（昇順の値のリスト）|


また、各ハイパーパラメータのスケーリングを指定することもできます。利用可能なスケーリングタイプは以下のとおりです。

|Scale Type|Description|Alias|
|--|--|--|
|UNIT_LINEAR_SCALE|実行可能空間を線形にスケーリングします| `'linear'`|
|UNIT_LOG_SCALE|実行可能空間を 0 から 1 まで対数的にスケーリングします。実行可能空間全体が厳密に正である必要があります。| `'log'`|
|UNIT_REVERSE_LOG_SCALE| 実行可能空間を 0 から 1 まで「逆」対数的にスケーリングします。その結果、実行可能空間の上部に近い値は、下部近くの点よりも広く分散されます。実行可能空間全体が厳密に正である必要があります。| `'reverse_log'`|

#### `max_trial_count`

サービスに実行を許可するトライアル数を決定し、HyperparameterTuningJob オブジェクトに maxTrialCount の値を設定します。

トライアルの数を増やすと一般的にはより良い結果が得られますが、常にそうとは限りません。通常は収穫逓減点が存在し、それを超えるとトライアルを追加しても精度にほとんど、またはまったく影響を与えなくなります。

#### `並列トライアル（Parallel trials）`
HyperparameterTuningJob で parallelTrialCount を設定することで、並行して実行できるトライアルの数を指定できます。

トライアルを並行して実行すると、トレーニングジョブにかかる時間（実時間。必要な総処理時間は通常変わりません）を短縮できるという利点があります。

**ただし、Google Vizier を使用する場合、並行して実行するとチューニングジョブ全体の効果が低下する可能性があります**。これは、Google Vizier が以前のトライアルの結果を使用して後続のトライアルのハイパーパラメータに割り当てる値を決定するためです。並行して実行すると、現在実行中のトライアルの結果を利用できないまま開始されるトライアルが発生します。


詳細については、[ドキュメント](https://cloud.google.com/vertex-ai/docs/training/hyperparameter-tuning-overview#hyperparameters) を参照してください。

#### 演習

以下の TODO を完了してください。

 - ハイパーパラメータ調整の指標タグと目標（goal）を指定する
 - 利用可能な[ハイパーパラメータ型](https://cloud.google.com/vertex-ai/docs/training/hyperparameter-tuning-overview#data-types) のいずれかを使用して、学習率、バッチサイズ、バケット数のハイパーパラメータ調整構成を指定する

In [ ]:
hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=f"taxifare_{timestamp}",
    custom_job=custom_job,
    metric_spec={...}, # TODO
    parameter_spec={
        'lr': hpt.DoubleParameterSpec(...), # TODO
        'nbuckets': hpt.IntegerParameterSpec(...) # TODO,
        'batch_size': ... # TODO
    },
    max_trial_count=8,
    parallel_trial_count=2,
)

すべての構成が完了したので、ジョブを送信して完了を待ちましょう。以下に表示される URL リンクを使用して、コンソールページにアクセスできます。

In [ ]:
hpt_job.run(sync=False)

**注:** 参考として、YAML ファイルで構成を指定できる同等の `gcloud ai` コマンドを以下に示します。

### TensorBoard を開く
ハイパーパラメータ調整ジョブで TensorBoard を使用して、各トライアルの損失曲線を比較できます。

ジョブが開始されるまでのしばらくの間はデータが表示されない場合があります。コンソールでジョブのステータスを確認し、ここに戻って右上の更新ボタンをクリックして TensorBoard を更新してください。

In [ ]:
%tensorboard --logdir {base_path}/trained_model_{timestamp} --port 8002

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License